In [1]:
import zipfile
from fastkml import kml

In [9]:
kmz_path = "D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz"


In [15]:
import zipfile, tempfile, os
from fastkml import kml
from fastkml.kml import Placemark
import geopandas as gpd
from shapely.geometry import shape

def iter_children(obj):
    kids = getattr(obj, "features", None)
    if callable(kids):
        return list(kids())
    if isinstance(kids, (list, tuple)):
        return list(kids)
    return []

def walk(obj):
    yield obj
    for c in iter_children(obj):
        yield from walk(c)

def kmz_to_gdf_robust(kmz_path):
    # 1) Read KML bytes from the KMZ (pick the first .kml by default)
    with zipfile.ZipFile(kmz_path, "r") as z:
        kml_candidates = [f for f in z.namelist() if f.lower().endswith(".kml")]
        if not kml_candidates:
            raise ValueError("No .kml found inside KMZ.")
        kml_name = sorted(kml_candidates, key=lambda s: (s!='doc.kml', len(s)))[0]  # prefer doc.kml
        kml_bytes = z.read(kml_name)

    # 2) Try fastkml first
    rows = []
    try:
        K = kml.KML()
        K.from_string(kml_bytes)  # IMPORTANT: bytes, not str

        n_feat = n_pm = n_geom = 0
        for feat in walk(K):
            n_feat += 1
            if isinstance(feat, Placemark):
                n_pm += 1
            geom = getattr(feat, "geometry", None)
            if geom is not None:
                n_geom += 1
                # fastkml geometry is shapely-like; ensure shapely instance if needed
                geom = geom if hasattr(geom, "geom_type") else shape(geom)
                rows.append({
                    "name": getattr(feat, "name", None),
                    "description": getattr(feat, "description", None),
                    "geometry": geom,
                })

        if rows:
            print(f"[fastkml] features={n_feat}, placemarks={n_pm}, with_geometry={n_geom}")
            return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
        else:
            print(f"[fastkml] features={n_feat}, placemarks={n_pm}, with_geometry={n_geom} => no geometries found.")
    except Exception as e:
        print(f"[fastkml] parsing failed: {e}")

    # 3) Fallback: extract KML and read via GeoPandas/Fiona (reads each Folder as a layer)
    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(kmz_path, "r") as z:
            z.extractall(tmp)
        kml_path = os.path.join(tmp, kml_name)

        # List available layers (KML driver treats each Folder as a layer)
        layers = gpd.io.file.fiona.listlayers(kml_path)
        print(f"[gpd] KML layers detected: {layers}")

        gdfs = []
        for lyr in layers:
            try:
                gdf = gpd.read_file(kml_path, driver="KML", layer=lyr)
                if not gdf.empty and "geometry" in gdf.columns:
                    gdfs.append(gdf)
            except Exception as e:
                print(f"[gpd] layer '{lyr}' failed to read: {e}")

        if gdfs:
            out = gpd.pd.concat(gdfs, ignore_index=True)
            # ensure GeoDataFrame and CRS
            out = gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")
            return out

    # 4) Last resort: return an empty GeoDataFrame (so caller won’t crash)
    print("[loader] No geometries found via fastkml or GeoPandas. Returning empty GeoDataFrame.")
    return gpd.GeoDataFrame(gpd.pd.DataFrame(columns=["name","description","geometry"]),
                            geometry="geometry", crs="EPSG:4326")


In [16]:
gdf = kmz_to_gdf_robust("D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz")
print(gdf.shape, gdf.dtypes)
print(gdf.head())


[fastkml] features=1, placemarks=0, with_geometry=0 => no geometries found.


AttributeError: 'NoneType' object has no attribute 'listlayers'

In [18]:
import zipfile
from fastkml import kml

def peek_kmz(kmz_path):
    with zipfile.ZipFile(kmz_path, "r") as z:
        kml_names = [f for f in z.namelist() if f.lower().endswith(".kml")]
        if not kml_names:
            print("No .kml inside.")
            return
        kml_bytes = z.read(kml_names[0])

    K = kml.KML(); K.from_string(kml_bytes)

    def iter_children(obj):
        kids = getattr(obj, "features", None)
        if callable(kids): return list(kids())
        if isinstance(kids,(list,tuple)): return list(kids)
        return []

    def walk(obj):
        yield obj
        for c in iter_children(obj):
            yield from walk(c)

    has_pm = has_overlay = has_link = False
    for f in walk(K):
        tn = type(f).__name__
        if tn == "Placemark": has_pm = True
        if tn in ("GroundOverlay","ScreenOverlay"): has_overlay = True
        if tn == "NetworkLink": has_link = True
        print(tn, getattr(f, "name", None))
    print(f"\nSummary: Placemark={has_pm}, Overlay={has_overlay}, NetworkLink={has_link}")

peek_kmz(r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz")


KML None

Summary: Placemark=False, Overlay=False, NetworkLink=False


In [19]:
import zipfile, os
from fastkml import kml
from fastkml.kml import Placemark
import geopandas as gpd
from shapely.geometry import shape

def iter_children(obj):
    kids = getattr(obj, "features", None)
    if callable(kids): return list(kids())
    if isinstance(kids, (list, tuple)): return list(kids)
    return []

def walk(obj):
    yield obj
    for c in iter_children(obj):
        yield from walk(c)

def inspect_kmz(kmz_path):
    print(f"== Inspecting: {kmz_path}")
    with zipfile.ZipFile(kmz_path, "r") as z:
        names = z.namelist()
        print("\n-- Archive contents --")
        for n in names:
            print(" ", n)
        kml_files = [n for n in names if n.lower().endswith(".kml")]
        if not kml_files:
            print("\nNo .kml files inside this KMZ.")
            return gpd.GeoDataFrame(columns=["name","description","geometry"], geometry="geometry", crs="EPSG:4326")

        rows = []
        for kml_name in kml_files:
            print(f"\n-- Parsing KML: {kml_name} --")
            kml_bytes = z.read(kml_name)
            K = kml.KML()
            try:
                K.from_string(kml_bytes)
            except Exception as e:
                print("  [parse error]", e)
                continue

            n_feat = n_pm = n_geom = 0
            for feat in walk(K):
                n_feat += 1
                if isinstance(feat, Placemark): n_pm += 1
                geom = getattr(feat, "geometry", None)
                if geom is not None:
                    n_geom += 1
                    geom = geom if hasattr(geom, "geom_type") else shape(geom)
                    rows.append({
                        "name": getattr(feat, "name", None),
                        "description": getattr(feat, "description", None),
                        "geometry": geom,
                    })
            print(f"  features={n_feat}, placemarks={n_pm}, with_geometry={n_geom}")

        if rows:
            gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
            print(f"\nCollected {len(gdf)} geometries from {len(kml_files)} KML file(s).")
            return gdf
        else:
            print("\nNo vector geometries found in any KML inside this KMZ.")
            return gpd.GeoDataFrame(columns=["name","description","geometry"], geometry="geometry", crs="EPSG:4326")

# Usage:
gdf = inspect_kmz(r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz")
print("\nGDF shape:", gdf.shape)


== Inspecting: D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz

-- Archive contents --
  doc.kml
  images/icon-36.png
  images/icon-20.png
  images/icon-38.png
  images/icon-108.png
  images/icon-58.png
  images/icon-89.png
  images/icon-37.png
  images/icon-102.png
  images/icon-26.png
  images/icon-61.png
  images/icon-41.png
  images/icon-114.png
  images/icon-11.png
  images/icon-70.png
  images/icon-4.png
  images/icon-75.png
  images/icon-2.png
  images/icon-63.png
  images/icon-29.png
  images/icon-115.png
  images/icon-119.png
  images/icon-125.png
  images/icon-77.png
  images/icon-16.png
  images/icon-71.png
  images/icon-83.png
  images/icon-112.png
  images/icon-31.png
  images/icon-117.png
  images/icon-79.png
  images/icon-107.png
  images/icon-60.png
  images/icon-66.png
  images/icon-132.png
  images/icon-69.png
  images/icon-73.png
  images/icon-124.png
  images/icon-46.png
  images/icon-127.png
  images/icon-8

In [20]:
import zipfile
kmz = r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz"
with zipfile.ZipFile(kmz) as z:
    txt = z.read("doc.kml").decode("utf-8", errors="replace")
print(txt[:2000])  # skim first ~2KB


<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
  <Document>
    <name>Ukraine Control Map</name>
    <description><![CDATA[Ukraine Control Map continuously updated by people at Project Owl OSINT (@projectowlosint)<br><br>Contact us on Twitter = @UAControlMap<br><br>Discord: https://discord.com/invite/projectowl<br><br>Information:<br>* Presence = reports of troops but no specific geolocation or imagery; covers a very wide area<br>* Position = there is some sort of evidence and/or geolocation of troops in question; covers a smaller, more specific area<br><br>Unit Icons: Unit was last sighted in that area. Click on unit card for details.<br><br>Yellow: UA advances<br>Purple: RU advances<br><br>Unit Patch Database: https://owlunits.com<br><br>Unit symbol guide: https://en.wikipedia.org/wiki/NATO_Joint_Military_Symbology<br><br>Changelog:  https://docs.google.com/spreadsheets/d/1u0N-CJNyuB4oSRwHpPkI27bAzwjCfBbAPUH4DKO7gqY (NEW URL as of August 2023)]]><

In [21]:
import zipfile, xml.etree.ElementTree as ET


with zipfile.ZipFile(kmz) as z:
    xml = z.read("doc.kml")  # bytes
ns = {"kml":"http://www.opengis.net/kml/2.2"}

root = ET.fromstring(xml)
def count(xpath): return len(root.findall(xpath, ns))

print("Placemark:", count(".//kml:Placemark"))
print("Folder:", count(".//kml:Folder"))
print("GroundOverlay:", count(".//kml:GroundOverlay"))
print("NetworkLink:", count(".//kml:NetworkLink"))
print("Style:", count(".//kml:Style"))
print("StyleMap:", count(".//kml:StyleMap"))


Placemark: 15132
Folder: 9
GroundOverlay: 0
NetworkLink: 0
Style: 190
StyleMap: 29


In [2]:
# kmz_to_geojson.py
# Pure-Python KMZ → GeoJSON (no GDAL/Fiona needed). Requires shapely.

import os, json, zipfile
import xml.etree.ElementTree as ET

from shapely.geometry import (
    Point, LineString, Polygon, MultiPoint, MultiLineString, MultiPolygon, GeometryCollection
)
from shapely.geometry import mapping

NS = {
    "kml": "http://www.opengis.net/kml/2.2",
    "gx":  "http://www.google.com/kml/ext/2.2",
}

def _coords_from_text(txt: str):
    """Parse KML coordinates text into [(lon, lat), ...]."""
    pts = []
    if not txt:
        return pts
    for chunk in txt.strip().split():
        parts = chunk.split(",")
        if len(parts) >= 2:
            lon = float(parts[0]); lat = float(parts[1])
            pts.append((lon, lat))
    return pts

def _parse_point(el):
    c = el.find(".//kml:Point/kml:coordinates", NS)
    if c is not None and c.text:
        pts = _coords_from_text(c.text)
        if pts:
            return Point(pts[0])
    return None

def _parse_linestring(el):
    c = el.find(".//kml:LineString/kml:coordinates", NS)
    if c is not None and c.text:
        pts = _coords_from_text(c.text)
        if len(pts) >= 2:
            return LineString(pts)
    return None

def _parse_polygon(el):
    outer = el.find(".//kml:Polygon/kml:outerBoundaryIs/kml:LinearRing/kml:coordinates", NS)
    if outer is None or outer.text is None:
        return None
    shell = _coords_from_text(outer.text)
    if len(shell) < 3:
        return None
    holes = []
    for inner in el.findall(".//kml:Polygon/kml:innerBoundaryIs/kml:LinearRing/kml:coordinates", NS):
        if inner.text:
            ring = _coords_from_text(inner.text)
            if len(ring) >= 3:
                holes.append(ring)
    return Polygon(shell, holes)

def _parse_gx_track(el):
    # gx:Track as a path (LineString). Ignores timestamps and altitude.
    coords = []
    for node in el.findall(".//gx:Track/gx:coord", NS):
        if node.text:
            parts = node.text.strip().split()
            if len(parts) >= 2:
                lon = float(parts[0]); lat = float(parts[1])
                coords.append((lon, lat))
    if len(coords) >= 2:
        return LineString(coords)
    return None

def _parse_multigeometry(el):
    """Parse MultiGeometry by collecting child geometries; return Multi* or GeometryCollection."""
    geoms = []

    # Points
    for c in el.findall(".//kml:MultiGeometry/kml:Point/kml:coordinates", NS):
        pts = _coords_from_text(c.text or "")
        if pts:
            geoms.append(Point(pts[0]))

    # LineStrings
    for c in el.findall(".//kml:MultiGeometry/kml:LineString/kml:coordinates", NS):
        pts = _coords_from_text(c.text or "")
        if len(pts) >= 2:
            geoms.append(LineString(pts))

    # Polygons
    for poly in el.findall(".//kml:MultiGeometry/kml:Polygon", NS):
        # Build a temporary element so we can reuse _parse_polygon
        tmp = ET.Element("tmp"); tmp.append(poly)
        g = _parse_polygon(tmp)
        if g is not None:
            geoms.append(g)

    if not geoms:
        return None

    types = {g.geom_type for g in geoms}
    if types == {"Point"}:
        return MultiPoint([g.coords[0] for g in geoms])
    if types == {"LineString"}:
        return MultiLineString([g.coords for g in geoms])
    if types == {"Polygon"}:
        return MultiPolygon([g for g in geoms])
    return GeometryCollection(geoms)

def _placemark_geometry(pm):
    """Try common geometry encodings in order."""
    for parser in (_parse_point, _parse_linestring, _parse_polygon, _parse_multigeometry, _parse_gx_track):
        g = parser(pm)
        if g is not None:
            return g
    return None

def _text(el, path):
    node = el.find(path, NS)
    return node.text.strip() if (node is not None and node.text) else None

def _walk_folders(el, path_acc=()):
    """Yield (Placemark element, folder_path_string) by descending Documents/Folders."""
    # If this is a Document/Folder, extend the path with its name
    tag_local = el.tag.split("}")[-1]
    if tag_local in ("Document", "Folder"):
        nm = _text(el, "kml:name")
        path_acc = path_acc + ((nm or "").strip(),)

    # Placemarks at this level
    for pm in el.findall("kml:Placemark", NS):
        yield pm, "/".join([p for p in path_acc if p])

    # Recurse into nested Folders/Documents
    for child in el.findall("kml:Folder", NS) + el.findall("kml:Document", NS):
        yield from _walk_folders(child, path_acc)

def kmz_to_geojson(kmz_path: str, out_path_or_dir: str = None):
    """Convert KMZ to GeoJSON. If out_path_or_dir is a directory, auto-name the file."""
    # Collect features from all KML files inside the KMZ
    features = []
    with zipfile.ZipFile(kmz_path, "r") as z:
        kml_files = [n for n in z.namelist() if n.lower().endswith(".kml")]
        if not kml_files:
            raise ValueError("No .kml found inside KMZ.")
        for kml_name in kml_files:
            xml_bytes = z.read(kml_name)
            root = ET.fromstring(xml_bytes)  # bytes → Element
            doc = root.find("kml:Document", NS) or root  # fall back to root if no Document
            for pm, folder_path in _walk_folders(doc, ()):
                geom = _placemark_geometry(pm)
                if geom is None:
                    continue
                feat = {
                    "type": "Feature",
                    "geometry": mapping(geom),
                    "properties": {
                        "name": _text(pm, "kml:name"),
                        "description": _text(pm, "kml:description"),
                        "styleUrl": _text(pm, "kml:styleUrl"),
                        "folder_path": folder_path
                    }
                }
                features.append(feat)

    # Decide output path
    if out_path_or_dir is None:
        base = os.path.splitext(os.path.basename(kmz_path))[0]
        out_geojson = os.path.join(os.path.dirname(kmz_path), f"{base}.geojson")
    else:
        if os.path.isdir(out_path_or_dir):
            base = os.path.splitext(os.path.basename(kmz_path))[0]
            out_geojson = os.path.join(out_path_or_dir, f"{base}.geojson")
        else:
            # Treat as explicit file path
            out_geojson = out_path_or_dir

    os.makedirs(os.path.dirname(out_geojson), exist_ok=True)

    fc = {"type": "FeatureCollection", "features": features}
    with open(out_geojson, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False)

    print(f"KMZ: {kmz_path}")
    print(f"Exported GeoJSON: {out_geojson}")
    print(f"Features written: {len(features)}")
    return out_geojson, len(features)


if __name__ == "__main__":
    # Your exact paths:
    kmz_in = r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz"
    out_dir = r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz"

    out_path, n = kmz_to_geojson(kmz_in, out_dir)
    # Optional: print an extra hint if nothing was found
    if n == 0:
        print("Warning: 0 features parsed. This KMZ may not contain vector geometries,")
        print("or it may use a geometry construct not covered by this parser.")


KMZ: D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/UAControlMapBackups-master/240318_0350.kmz
Exported GeoJSON: D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz\240318_0350.geojson
Features written: 15132


In [3]:
import geopandas as gpd
import folium
from folium.plugins import FastMarkerCluster, HeatMap

# === Path to the GeoJSON you just exported ===
GEOJSON_PATH = r"D:/Projects/GitRepositories/TheAWOLthing/AWOL_Research/data/kmz/240318_0350.geojson"

# Load
gdf = gpd.read_file(GEOJSON_PATH)

# Make the base map (centered & zoomed to data bounds)
minx, miny, maxx, maxy = gdf.total_bounds
m = folium.Map(tiles="cartodbpositron")
m.fit_bounds([[miny, minx], [maxy, maxx]])

# Split by geometry type for styling/perf
points   = gdf[gdf.geometry.geom_type == "Point"]
lines    = gdf[gdf.geometry.geom_type.isin(["LineString","MultiLineString"])]
polygons = gdf[gdf.geometry.geom_type.isin(["Polygon","MultiPolygon"])]

# ---- Polylines / polygons (draw as GeoJSON layers) ----
if not lines.empty:
    folium.GeoJson(
        lines.__geo_interface__,
        name="Lines",
        style_function=lambda x: {"weight": 2, "opacity": 0.8}
    ).add_to(m)

if not polygons.empty:
    folium.GeoJson(
        polygons.__geo_interface__,
        name="Polygons",
        style_function=lambda x: {"weight": 1, "fillOpacity": 0.15}
    ).add_to(m)

# ---- Points (choose ONE of the two options) ----
if not points.empty:
    # Option A: very fast clusters (no popups)
    coords = [[geom.y, geom.x] for geom in points.geometry]
    FastMarkerCluster(coords, name="Points (clustered)").add_to(m)

    # Option B (optional): heatmap (good for density overview)
    # HeatMap(coords, name="Heatmap", radius=8).add_to(m)

# Layer control
folium.LayerControl().add_to(m)

# Save and open
m.save("uacontrolmap_preview.html")
print("Saved: uacontrolmap_preview.html")


Saved: uacontrolmap_preview.html
